# Notebook 01: Download `DocLayNet`
**Author**: Juan Pablo Triana Martinez
**Date**: 2026-03-12

One of the projects I am working on is to create an **optical character recognition** system. This is divided into the main steps

```Python
pdf_doc -> preprocessed_img -> text_detection -> text_recognition
```
**Before attempting to recognize text**, we need to preprocess the data, as well as obtain boundary boxes where the text is located in order to increase the reliability of the **optical character recogntion system**.

Therefore, this notebook would focus into performing **Text Detection** on the following dataset:

*DocLayNet provides page-by-page layout segmentation ground-truth using bounding-boxes for 11 distinct class labels on 80863 unique pages from 6 document categories.*

- **DocLayNet Datast link**: https://arxiv.org/pdf/2206.01062
- **Github link**: https://github.com/DS4SD/DocLayNet

![](https://evermap.com/UserGuides/T1/Tutorial_ABM_CharacterRecognitionInPDF%20(8).PNG)

### `DocLayNet` Dataset

This dataset contains around 80863 PDFs pages. Across these, we also have 7059 carry two instances of human annotaions, and 1591 carry three. Having a total of 91104 total annotations. We have the following categories, which are (found in **Table 1** from DocLayNet paper):
* Caption ~ 22524
* Footnote ~ 6318
* Formula ~ 25027
* List-item ~ 185660
* Page-footer ~ 70878
* Page-header ~ 58022
* Picture ~ 45976
* Section-header ~ 142884
* Table ~ 34733
* Text ~ 510377
* Title ~ 5071
* **Image**: PIL image of size 1025 y 1025.

![](https://miro.medium.com/v2/resize:fit:1100/format:webp/1*VokPPkMBfMO-RYHEi7BG6g.png)

This dataset contains rectangular bounding boxes using COCO format, as shown here: https://cocodataset.org/#format-data

*Without further do, let's prepare the codes in order to download the `DocLayNet` dataset.*

## 1. Create a `download_raw_data` and `extract_raw_data` methods to obtain DocLayNet 

In [7]:
from pathlib import Path
from tqdm import tqdm
import requests

def download_raw_data(
    data_folder_path: Path,
    filename_link:str,
    chunk_size:int = 16) -> Path:
    '''
    The following function will download the raw data from a privided link and save it in the
    specified data_folder_path
    Args:
        data_folder_path (Path): Path where the raw data will be saved.
        filename_link (str): The link to the raw data file.
    '''

    # Create the data_folder_path if it doesn't exist
    data_folder_path.mkdir(parents=True, exist_ok=True)

    # Let's create a raw folder inside the data folder
    raw_folder_path = data_folder_path / "raw"
    raw_folder_path.mkdir(parents=True, exist_ok=True)

    # Download the file and save it in the raw folder
    file_name = filename_link.split("/")[-1] # Select the last split, filename
    file_path = raw_folder_path / file_name

    if file_path.exists():
        print(f"[INFO] File already exists: {file_path}")
        return file_path
    else:
        print(f"[INFO] Downloading raw data from: {filename_link}")

        #Stream Donbwload the file
        with requests.get(filename_link, stream=True) as response:
            response.raise_for_status()

            # Find the total file size and chunk size
            total_size = int(response.headers.get('content-length', 0))
            chunk_size = chunk_size * 1024 * 1024 # 16 MB

            # Open the file and write the chunks to it
            with open(file_path, 'wb') as f, tqdm(
                total=total_size,
                unit='B',
                unit_scale=True,
                unit_divisor=1024,
                desc=file_name
            ) as pbar:
                for chunk in response.iter_content(chunk_size=chunk_size):
                    if chunk:
                        f.write(chunk)
                        pbar.update(len(chunk))
    print(f"[INFO] Raw data downloaded at: {file_path}")
    return file_path

### 1.1 Get the links for `DocLayNet_core.zip` and `DocLayNet_extra.zip`

Both files contain the following, (taken from the README.md):
1. PNG images of all pages, resized to square 1025 x 1025px
2. Bounding-box annotations in COCO format for each PNG image
3. Extra: Single-page PDF files matching each PNG image
4. Extra: JSON file matching each PDF page, which provides the digital text cells with coordinates and content

In [3]:
# Links for DocLayNet dataset
internet_links = {
    "core": "https://codait-cos-dax.s3.us.cloud-object-storage.appdomain.cloud/dax-doclaynet/1.0.0/DocLayNet_core.zip",
    "extra": "https://codait-cos-dax.s3.us.cloud-object-storage.appdomain.cloud/dax-doclaynet/1.0.0/DocLayNet_extra.zip"
}

In [ ]:
# Setup the data folder path
data_folder_path = Path().cwd().parent / "data"

### 1.2 Download `DocLayNet_core.zip`

In [15]:
doclaynet_core_path = download_raw_data(
    data_folder_path=data_folder_path,
    filename_link=internet_links["core"])

[INFO] File already exists: c:\Users\ajedr\Documents\Masters_Post_Cert_AI_Stanford_USD_2023_2026\AAI_590_Machine_Learning_Capstone\AAI-590-OCR-Master\data\raw\DocLayNet_core.zip


### 1.3 Download `DocLayNet_extra.zip`

In [16]:
doclaynet_extra_path = download_raw_data(
    data_folder_path=data_folder_path,
    filename_link=internet_links["extra"])

[INFO] File already exists: c:\Users\ajedr\Documents\Masters_Post_Cert_AI_Stanford_USD_2023_2026\AAI_590_Machine_Learning_Capstone\AAI-590-OCR-Master\data\raw\DocLayNet_extra.zip


## 2. Let's create the `extract_raw_data` method

Now that we have both our `DocLayNet_core` and `DocLayNet_extra` zip folders, we need to know extract them

In [14]:
import zipfile

def extract_raw_data(
    data_folder_path:Path,
    zip_file_data_path: Path) -> Path:
    '''
    The following function will extract the raw data from a zip file
    into a specified data folder path
    Args:
        data_folder_path (Path): Path where the extracted data will be saved.
        zip_file_data_path (Path): Path where the raw data zip file is located.
    '''

    # Create a folder for extracted data
    extracted_data_path = data_folder_path / "extracted"
    extracted_data_path.mkdir(parents = True, exist_ok=True)

    # Let's obtain the zip local file name
    file_name = zip_file_data_path.stem

    # Let's now extract the data from zipfile
    with zipfile.ZipFile(zip_file_data_path, "r") as zip_ref:
        s = zip_ref.infolist()

        # Iterate over the zip file contents and extract them
        for member in tqdm(s, desc="Extracting files", total = (len(s))):
            output_path = extracted_data_path / file_name / member.filename
            if output_path.exists():
                continue
            zip_ref.extract(member, extracted_data_path/ file_name)
    return extracted_data_path / file_name


### 2.1 Extract `DocLayNet_core`

In [17]:
doclaynet_core_extracted_path = extract_raw_data(
    data_folder_path=data_folder_path,
    zip_file_data_path=doclaynet_core_path)

Extracting files: 100%|██████████| 81478/81478 [06:02<00:00, 224.61it/s]


### 2.2 Extract `DocLayNet_extra`

In [18]:
doclaynet_extra_extracted_path = extract_raw_data(
    data_folder_path = data_folder_path,
    zip_file_data_path = doclaynet_extra_path
)

Extracting files: 100%|██████████| 162946/162946 [04:18<00:00, 629.32it/s]
